# Control Panel 3: Test Model

This notebook evaluates the trained model and visualizes its performance.

## Usage
1. Ensure model has been trained using `2_Train_Model.ipynb`
2. Run all cells to evaluate and visualize results

In [ ]:
# Import required modules
import sys
from pathlib import Path

# Add src to path
sys.path.append('src')

from src.config import default_params_5, DATA_DIR, MODEL_DIR, RESULTS_DIR
from src.dataloader import create_dataloader, get_dataset_info
from src.model import get_model

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import h5py

## Load Trained Model

In [ ]:
# Configuration
config = {
    'model_type': 'autoencoder',  # Must match training
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'batch_size': 32,
    'num_workers': 4,
}

# Load model
model = get_model(model_type=config['model_type'])
model = model.to(config['device'])

# Load checkpoint
checkpoint_path = Path(MODEL_DIR) / 'best_model.pth'

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Model not found at {checkpoint_path}. "
        "Please run 2_Train_Model.ipynb first."
    )

checkpoint = torch.load(checkpoint_path, map_location=config['device'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded model from epoch {checkpoint['epoch']}")
print(f"Best training loss: {checkpoint['loss']:.6f}")
print(f"Using device: {config['device']}")

## Load Test Data

In [ ]:
# Load dataset
dataset_path = Path(DATA_DIR) / 'dqd_dataset.h5'

if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {dataset_path}. "
        "Please run 1_Generate_Data.ipynb first."
    )

# Create dataloader
test_loader = create_dataloader(
    str(dataset_path),
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=config['num_workers'],
    normalize=True
)

print(f"Test dataset loaded: {len(test_loader.dataset)} samples")

## Evaluate Model Performance

In [ ]:
# Evaluate on test set
criterion = nn.MSELoss()
test_loss = 0.0
all_predictions = []
all_targets = []
all_latents = []

print("Evaluating model...")

with torch.no_grad():
    for inputs, targets in tqdm(test_loader):
        inputs = inputs.to(config['device'])
        targets = targets.to(config['device'])
        
        if config['model_type'] == 'autoencoder':
            outputs, latents = model(inputs)
            all_latents.append(latents.cpu().numpy())
        else:
            outputs = model(inputs)
        
        loss = criterion(outputs, targets)
        test_loss += loss.item()
        
        all_predictions.append(outputs.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

# Calculate average loss
avg_test_loss = test_loss / len(test_loader)
print(f"\nAverage test loss: {avg_test_loss:.6f}")

# Concatenate all results
all_predictions = np.concatenate(all_predictions, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

if config['model_type'] == 'autoencoder':
    all_latents = np.concatenate(all_latents, axis=0)
    print(f"Latent space shape: {all_latents.shape}")

## Visualize Reconstructions

Compare original images with model reconstructions.

In [ ]:
# Visualize reconstructions
num_samples = 8
indices = np.random.choice(len(all_predictions), num_samples, replace=False)

fig, axes = plt.subplots(2, num_samples, figsize=(20, 5))

for i, idx in enumerate(indices):
    # Original
    axes[0, i].imshow(all_targets[idx, 0], cmap='hot', aspect='auto')
    axes[0, i].set_title(f'Original {idx}')
    axes[0, i].axis('off')
    
    # Reconstruction
    axes[1, i].imshow(all_predictions[idx, 0], cmap='hot', aspect='auto')
    axes[1, i].set_title(f'Reconstructed {idx}')
    axes[1, i].axis('off')

plt.suptitle('Original vs Reconstructed Stability Diagrams', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Reconstruction Error Analysis

In [ ]:
# Calculate reconstruction errors
errors = np.mean((all_predictions - all_targets) ** 2, axis=(1, 2, 3))

# Plot histogram of errors
plt.figure(figsize=(10, 6))
plt.hist(errors, bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Mean Squared Error')
plt.ylabel('Frequency')
plt.title('Distribution of Reconstruction Errors')
plt.axvline(errors.mean(), color='r', linestyle='--', 
            label=f'Mean: {errors.mean():.6f}')
plt.axvline(np.median(errors), color='g', linestyle='--', 
            label=f'Median: {np.median(errors):.6f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Reconstruction error statistics:")
print(f"  Mean: {errors.mean():.6f}")
print(f"  Std:  {errors.std():.6f}")
print(f"  Min:  {errors.min():.6f}")
print(f"  Max:  {errors.max():.6f}")

## Visualize Best and Worst Reconstructions

In [ ]:
# Find best and worst reconstructions
best_indices = np.argsort(errors)[:4]
worst_indices = np.argsort(errors)[-4:]

fig, axes = plt.subplots(4, 4, figsize=(12, 12))

# Best reconstructions
for i, idx in enumerate(best_indices):
    axes[i, 0].imshow(all_targets[idx, 0], cmap='hot', aspect='auto')
    axes[i, 0].set_title(f'Best #{i+1} Original')
    axes[i, 0].axis('off')
    
    axes[i, 1].imshow(all_predictions[idx, 0], cmap='hot', aspect='auto')
    axes[i, 1].set_title(f'Recon (MSE: {errors[idx]:.4f})')
    axes[i, 1].axis('off')

# Worst reconstructions
for i, idx in enumerate(worst_indices):
    axes[i, 2].imshow(all_targets[idx, 0], cmap='hot', aspect='auto')
    axes[i, 2].set_title(f'Worst #{i+1} Original')
    axes[i, 2].axis('off')
    
    axes[i, 3].imshow(all_predictions[idx, 0], cmap='hot', aspect='auto')
    axes[i, 3].set_title(f'Recon (MSE: {errors[idx]:.4f})')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()

## Latent Space Visualization

If using an autoencoder, visualize the learned latent representations.

In [ ]:
if config['model_type'] == 'autoencoder' and len(all_latents) > 0:
    from sklearn.decomposition import PCA
    from sklearn.manifold import TSNE
    
    # PCA visualization
    pca = PCA(n_components=2)
    latents_pca = pca.fit_transform(all_latents)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # PCA
    scatter = axes[0].scatter(latents_pca[:, 0], latents_pca[:, 1], 
                             c=errors, cmap='viridis', alpha=0.6, s=20)
    axes[0].set_xlabel('First Principal Component')
    axes[0].set_ylabel('Second Principal Component')
    axes[0].set_title('Latent Space (PCA)')
    plt.colorbar(scatter, ax=axes[0], label='Reconstruction Error')
    
    # t-SNE (on a subset for speed)
    n_subset = min(1000, len(all_latents))
    subset_indices = np.random.choice(len(all_latents), n_subset, replace=False)
    
    print("Computing t-SNE (this may take a moment)...")
    tsne = TSNE(n_components=2, random_state=42)
    latents_tsne = tsne.fit_transform(all_latents[subset_indices])
    
    scatter = axes[1].scatter(latents_tsne[:, 0], latents_tsne[:, 1], 
                             c=errors[subset_indices], cmap='viridis', 
                             alpha=0.6, s=20)
    axes[1].set_xlabel('t-SNE Dimension 1')
    axes[1].set_ylabel('t-SNE Dimension 2')
    axes[1].set_title(f'Latent Space (t-SNE, n={n_subset})')
    plt.colorbar(scatter, ax=axes[1], label='Reconstruction Error')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nPCA explained variance ratio: {pca.explained_variance_ratio_}")
    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")
else:
    print("Latent space visualization only available for autoencoder models.")

## Save Results

In [ ]:
# Create results directory
results_path = Path(RESULTS_DIR)
results_path.mkdir(exist_ok=True)

# Save evaluation metrics
results = {
    'test_loss': avg_test_loss,
    'error_mean': errors.mean(),
    'error_std': errors.std(),
    'error_min': errors.min(),
    'error_max': errors.max(),
}

# Save to file
import json
with open(results_path / 'test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {results_path / 'test_results.json'}")
print("\nTest Results Summary:")
for key, value in results.items():
    print(f"  {key}: {value:.6f}")

## Conclusion

✅ Model evaluation complete!

### Summary
- Model successfully evaluated on test dataset
- Reconstruction quality analyzed
- Latent space visualized
- Results saved for future reference

### Next Steps
- Fine-tune hyperparameters if needed
- Try different model architectures
- Generate more diverse training data
- Apply model to real experimental data